In [6]:
from datetime import datetime, timedelta

def parse_time(t):
    return datetime.strptime(t, "%Y-%m-%d %H:%M:%S")


def detect_bruteforce(events, threshold=5, window_minutes=2):
    """
    Detect brute-force login attempts:
    - At least 'threshold' failed logins (Event ID 4625)
    - Within 'window_minutes'
    - Report whether a successful login (Event ID 4624) followed.
    """

    # Sort events by timestamp
    events = sorted(events, key=lambda e: parse_time(e["timestamp"]))

    # Group events by account
    by_account = {}
    for e in events:
        by_account.setdefault(e["account"], []).append(e)

    results = {}

    # Analyze each account
    for account, acc_events in by_account.items():

        failures = [e for e in acc_events if e["event_id"] == 4625]
        successes = [e for e in acc_events if e["event_id"] == 4624]

        flagged = False

        for i in range(len(failures)):
            window_start = parse_time(failures[i]["timestamp"])
            window_end = window_start + timedelta(minutes=window_minutes)

            count = sum(
                1
                for f in failures
                if window_start <= parse_time(f["timestamp"]) <= window_end
            )

            if count >= threshold:
                flagged = True
                break

        if flagged:
            followed_by_success = any(
                parse_time(s["timestamp"]) > parse_time(failures[-1]["timestamp"])
                for s in successes
            )

            results[account] = {
                "failed_attempts": len(failures),
                "followed_by_success": followed_by_success,
                "source_ips": sorted({f["source_ip"] for f in failures}),
            }

    return results

In [7]:
events = [
    {"timestamp": "2026-08-04 10:00:00", "event_id": 4625, "account": "admin", "source_ip": "192.168.1.10"},
    {"timestamp": "2026-08-04 10:00:20", "event_id": 4625, "account": "admin", "source_ip": "192.168.1.10"},
    {"timestamp": "2026-08-04 10:00:40", "event_id": 4625, "account": "admin", "source_ip": "192.168.1.10"},
    {"timestamp": "2026-08-04 10:01:00", "event_id": 4625, "account": "admin", "source_ip": "192.168.1.10"},
    {"timestamp": "2026-08-04 10:01:20", "event_id": 4625, "account": "admin", "source_ip": "192.168.1.10"},
    {"timestamp": "2026-08-04 10:02:00", "event_id": 4624, "account": "admin", "source_ip": "192.168.1.10"},
]

print(detect_bruteforce(events))

{'admin': {'failed_attempts': 5, 'followed_by_success': True, 'source_ips': ['192.168.1.10']}}
